# Ejercicio 07 · Evaluación de impacto y auditoría de equidad

Expediente reproducible que integra LAB11 (evaluación de impacto ético) y LAB12 (auditoría de
equidad con Fairlearn).

**Alcance del caso.** El conjunto `data/territorial_risk_synthetic.csv` es **sintético y didáctico**:
no representa personas, zonas, tasas ni estadísticas reales de República Dominicana. Sirve para
practicar contratos de datos, evaluación temporal, auditoría de subgrupos y gobernanza. Ninguna cifra
de este notebook describe poblaciones reales.

**Marco de referencia.** Tres capas que no deben confundirse:

| Capa | Qué obliga | Qué es aquí |
|---|---|---|
| **Obligación legal** | Normas vinculantes (protección de datos, principios de igualdad) | Ninguna verificación específica se aplica al caso sintético; se documenta la ausencia de base legal y se prohíbe el uso con datos reales sin revisión jurídica |
| **Marco voluntario** | NIST AI RMF 1.0 (DOI 10.6028/NIST.AI.100-1), que organiza GOVERN, MAP, MEASURE y MANAGE | Es el marco que estructura el registro de riesgos y la auditoría |
| **Recomendación ética** | Juicio profesional sobre laanahELO y la proporcionalidad | La alternativa no predictiva, el recurso de apelación y la condición de retiro |

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
# El kernel arranca en la carpeta del notebook, pero los scripts del proyecto base resuelven
# el dataset con rutas relativas a la raiz del proyecto: nos situamos en ella.
import os

os.chdir(ROOT)

from inf8239_u04.data import features, load_data, temporal_split
from inf8239_u04.fairness import audit, intersectional
from inf8239_u04.model import build_model, fit_measure

REPORTS = ROOT / "reports"
print("Proyecto:", ROOT)

Proyecto: /home/trek/source/uasd/cienciadedatos/tareas/Unidad_4/unidad04-08


## 1. Contrato de datos y corte temporal

In [2]:
df = load_data()
train, test = temporal_split(df)
print("Filas x columnas:", df.shape)
print("Target:", df.target.value_counts(normalize=True).round(3).to_dict())
print("Grupos:", df.protected_group.value_counts().to_dict())
print("Bandas de edad:", df.age_band.value_counts().to_dict())
print("Corte temporal: entrenamiento hasta", train.received_date.max().date(),
      "| prueba desde", test.received_date.min().date(),
      f"({len(train)} / {len(test)} registros)")
print("Tasa de positivos por grupo (prueba):")
print((pd.crosstab(test.protected_group, test.target, normalize="index") * 100).round(1))

Filas x columnas: (640, 9)
Target: {1: 0.552, 0: 0.448}
Grupos: {'G1': 396, 'G2': 244}
Bandas de edad: {'60+': 179, '30-44': 163, '45-59': 150, '18-29': 148}
Corte temporal: entrenamiento hasta 2026-09-24 | prueba desde 2026-09-25 (480 / 160 registros)
Tasa de positivos por grupo (prueba):
target              0     1
protected_group            
G1               51.5  48.5
G2               27.1  72.9


## 2. LAB11 · Propósito, interesados y registro de riesgos

Los valores de la evaluación de impacto son decisiones del estudiante (archivos editables en
`student_inputs/`); el script del profesor los valida y calcula la prioridad de cada riesgo.

In [3]:
impacto = yaml.safe_load((ROOT / "student_inputs/lab11_impact.yml").read_text(encoding="utf-8"))
for clave in ["purpose", "assisted_decision", "non_predictive_alternative", "out_of_scope_use",
              "withdrawal_condition", "appeal_mechanism"]:
    print(f"{clave}: {impacto[clave][:180]}...")
print()
interesados = pd.read_csv(ROOT / "student_inputs/stakeholders.csv")
print(interesados[["stakeholder", "power", "participation"]].to_string(index=False))

purpose: Priorizar la revisión humana de casos con señal de riesgo territorial para orientar el esfuerzo de prevención (verificación en campo y acompañamiento). El sistema no decide: ordena...
assisted_decision: El DSS calcula una probabilidad de incidente y un orden de prioridad. La persona responsable de la jornada revisa la lista, puede abrir, posponer o descartar cada caso, y deja regi...
non_predictive_alternative: Reparto rotativo por zona y severidad reportada, priorización manual de todas las alertas recibidas, o inversión en infraestructura y tiempo comunitario priorizado por criterios pu...
out_of_scope_use: Prohibido usar el modelo o su ranking para: decidir sanciones, sanciones administrativas o prohibiciones de cualquier tipo; abrir expedientes o imputar responsabilidades; distribui...
withdrawal_condition: El sistema se retira o se detiene si ocurre cualquiera de estas condiciones: (1) la brecha de recall entre grupos supera la tolerancia acordada en la auditoria del LAB1

In [4]:
riesgos = pd.read_csv(ROOT / "reports/lab11/risk_register_scored.csv")
print("Riesgos registrados:", len(riesgos), "| con control y responsable:",
      int((riesgos["control"].astype(str).str.len() > 0).sum()),
      "| con evidencia:", int((riesgos["evidence"].astype(str).str.len() > 0).sum()))
print(riesgos[["id", "likelihood_1_4", "impact_1_4", "priority", "owner", "residual"]]
      .sort_values("priority", ascending=False).to_string(index=False))

figura, eje = plt.subplots(figsize=(9, 4.6))
colores = {"high": "#b91c1c", "medium": "#d97706", "low": "#65a30d"}
eje.barh(riesgos["id"], riesgos["priority"], color=[colores[r] for r in riesgos["residual"]])
eje.set_xlabel("Prioridad (probabilidad x impacto)")
eje.set_title("Riesgos priorizados · color = riesgo residual")
eje.invert_yaxis()
eje.grid(axis="x", alpha=0.3)
figura.tight_layout()
figura.savefig(REPORTS / "lab11/riesgos_priorizados.png", dpi=170, bbox_inches="tight")
plt.close(figura)

Riesgos registrados: 9 | con control y responsable: 9 | con evidencia: 9
 id  likelihood_1_4  impact_1_4  priority                                                                   owner residual
R03               3           3         9 Responsable de equidad (equipo de datos con validacion de la direccion)   medium
R01               2           4         8                            Analista de riesgo responsable de la jornada   medium
R02               2           4         8                                                Coordinador de operacion   medium
R07               2           4         8                                                  Direccion del programa     high
R05               2           3         6                                                  Direccion del programa   medium
R08               2           3         6                                                  Responsable de equidad   medium
R06               3           2         6                         

## 3. LAB12 · Métricas globales, por grupo e interseccionales

Dos alternativas sobre **la misma partición temporal y la misma semilla**: regresión logística y
árbol de decisión. La métrica de utilidad es el recall (detectar casos con incidente probable); la
métrica de daño es el FPR (alarmar sobre casos sin riesgo) y la cobertura del catálogo de alertas.

In [5]:
resultados = {}
for modelo in ["logistic", "tree"]:
    m = build_model(modelo)
    recursos = fit_measure(m, features(train), train.target, f"models/{modelo}.joblib")
    pred = m.predict(features(test))
    overall, grupos, diff = audit(test.target, pred, test.protected_group)
    _, inter, _ = audit(test.target, pred, intersectional(test))
    resultados[modelo] = {"overall": overall, "groups": grupos, "diff": diff, "inter": inter,
                          "resources": recursos}
    print(f"\n=== {modelo} ===")
    print("global:", {k: round(float(v), 4) for k, v in overall.items()})
    print(grupos.round(3).to_string())
    print("brechas:", {k: round(float(v), 4) for k, v in diff.items()})
    print("recursos:", recursos)


=== logistic ===
global: {'count': 160.0, 'accuracy': 0.7188, 'selection_rate': 0.5312, 'recall': 0.7174, 'precision': 0.7765, 'fpr': 0.2794}
                 count  accuracy  selection_rate  recall  precision    fpr
protected_group                                                           
G1               101.0     0.703           0.446   0.653      0.711  0.250
G2                59.0     0.746           0.678   0.791      0.850  0.375
brechas: {'count': 42.0, 'accuracy': 0.0428, 'selection_rate': 0.2324, 'recall': 0.1376, 'precision': 0.1389, 'fpr': 0.125}
recursos: {'training_seconds': 0.020140571999945678, 'model_kb': 3.5634765625}

=== tree ===
global: {'count': 160.0, 'accuracy': 0.7063, 'selection_rate': 0.5437, 'recall': 0.7174, 'precision': 0.7586, 'fpr': 0.3088}
                 count  accuracy  selection_rate  recall  precision    fpr
protected_group                                                           
G1               101.0     0.683           0.446   0.633      0.6

In [6]:
log = resultados["logistic"]["groups"]
tree = resultados["tree"]["groups"]
figura, (eje1, eje2) = plt.subplots(1, 2, figsize=(13, 4.8))
for eje, (nombre, tabla) in zip([eje1, eje2], [("Logística", log), ("Árbol", tree)]):
    x = np.arange(len(tabla))
    ancho = 0.35
    eje.bar(x - ancho / 2, tabla["recall"], ancho, label="recall", color="#0f7c7b")
    eje.bar(x + ancho / 2, tabla["fpr"], ancho, label="FPR", color="#b45309")
    for i, (r, f) in enumerate(zip(tabla["recall"], tabla["fpr"])):
        eje.text(i - ancho / 2, r + 0.02, f"{r:.2f}", ha="center", fontsize=8)
        eje.text(i + ancho / 2, f + 0.02, f"{f:.2f}", ha="center", fontsize=8)
    eje.set_xticks(x, tabla.index)
    eje.set_ylim(0, 1)
    eje.set_ylabel("tasa")
    eje.set_title(f"{nombre} · recall y FPR por grupo (n en la etiqueta)")
    eje.legend(fontsize=8)
    eje.grid(axis="y", alpha=0.25)
figura.tight_layout()
figura.savefig(REPORTS / "lab12/metricas_por_grupo.png", dpi=170, bbox_inches="tight")
plt.close(figura)
print("Figura guardada en reports/lab12/metricas_por_grupo.png")

Figura guardada en reports/lab12/metricas_por_grupo.png


In [7]:
inter = resultados["logistic"]["inter"].copy()
inter["soporte"] = np.where(inter["count"] < 15, "INSUFICIENTE", "ok")
figura, eje = plt.subplots(figsize=(10, 5))
colores = ["#b91c1c" if s == "INSUFICIENTE" else "#0f7c7b" for s in inter["soporte"]]
eje.bar(inter.index, inter["recall"], color=colores)
for i, (c, r) in enumerate(zip(inter["count"], inter["recall"])):
    eje.text(i, r + 0.02, f"n={int(c)}", ha="center", fontsize=8)
eje.set_ylabel("recall")
eje.set_title("Recall interseccional (grupo | banda de edad) · rojo = soporte insuficiente")
eje.tick_params(axis="x", rotation=30)
eje.set_ylim(0, 1.15)
eje.grid(axis="y", alpha=0.25)
figura.tight_layout()
figura.savefig(REPORTS / "lab12/recall_interseccional.png", dpi=170, bbox_inches="tight")
plt.close(figura)
print(inter[["count", "recall", "soporte"]].round(3).to_string())

                     count  recall       soporte
sensitive_feature_0                             
G1 | 18-29            18.0   0.750            ok
G1 | 30-44            24.0   0.562            ok
G1 | 45-59            35.0   0.588            ok
G1 | 60+              24.0   0.833            ok
G2 | 18-29            11.0   0.875  INSUFICIENTE
G2 | 30-44            14.0   0.667  INSUFICIENTE
G2 | 45-59             8.0   1.000  INSUFICIENTE
G2 | 60+              26.0   0.765            ok


## 4. Comparación de alternativas y decisión

Misma utilidad global, brechas distintas y costo distinto: ese es el trade-off que decide.

In [8]:
comparativa = pd.DataFrame([
    {
        "modelo": "Logística",
        "accuracy": float(resultados["logistic"]["overall"]["accuracy"]),
        "recall": float(resultados["logistic"]["overall"]["recall"]),
        "fpr": float(resultados["logistic"]["overall"]["fpr"]),
        "brecha_recall": float(resultados["logistic"]["diff"]["recall"]),
        "brecha_fpr": float(resultados["logistic"]["diff"]["fpr"]),
        "segundos": resultados["logistic"]["resources"]["training_seconds"],
        "kb": resultados["logistic"]["resources"]["model_kb"],
    },
    {
        "modelo": "Árbol",
        "accuracy": float(resultados["tree"]["overall"]["accuracy"]),
        "recall": float(resultados["tree"]["overall"]["recall"]),
        "fpr": float(resultados["tree"]["overall"]["fpr"]),
        "brecha_recall": float(resultados["tree"]["diff"]["recall"]),
        "brecha_fpr": float(resultados["tree"]["diff"]["fpr"]),
        "segundos": resultados["tree"]["resources"]["training_seconds"],
        "kb": resultados["tree"]["resources"]["model_kb"],
    },
]).round(4)
comparativa.to_csv(REPORTS / "lab12/comparativa_alternativas.csv", index=False)
print(comparativa.to_string(index=False))

figura, eje = plt.subplots(figsize=(8.5, 4.6))
eje.scatter(comparativa["brecha_recall"], comparativa["recall"], s=170, color="#0f7c7b")
for _, fila in comparativa.iterrows():
    eje.annotate(fila["modelo"], (fila["brecha_recall"], fila["recall"]),
                 textcoords="offset points", xytext=(8, 6), fontsize=9)
eje.set_xlabel("Brecha de recall entre grupos (menor es mejor)")
eje.set_ylabel("Recall global (mayor es mejor)")
eje.set_title("Utilidad frente a brecha: la logística domina al árbol")
eje.grid(alpha=0.25)
figura.tight_layout()
figura.savefig(REPORTS / "lab12/utilidad_vs_brecha.png", dpi=170, bbox_inches="tight")
plt.close(figura)

   modelo  accuracy  recall    fpr  brecha_recall  brecha_fpr  segundos     kb
Logística    0.7188  0.7174 0.2794         0.1376      0.1250    0.0201 3.5635
    Árbol    0.7062  0.7174 0.3088         0.1813      0.1683    0.0055 7.9014


In [9]:
decision = yaml.safe_load((ROOT / "student_inputs/lab12_decision.yml").read_text(encoding="utf-8"))
for clave, valor in decision.items():
    print(f"\n[{clave}]\n{valor}")


[priority_harm]
Falso negativo: que un caso con incidente probable no llegue a la cola priorizada y la atencion preventiva no se vea a tiempo (las personas que debian recibirla quedan sin verificar). En segundo lugar, el falso positivo que desplaza capacidad de campo hacia casos sin riesgo.

[primary_metric]
Brecha de recall (TPR) entre grupos, junto al recall absoluto del grupo con peor desempeño. El recall global 0.7174 no basta: en el grupo G1 el recall es 0.653 frente a 0.791 en G2.

[tolerance]
Decision documentada: LIMITAR (no hay despliegue autonomo; piloto en modo REVIEW con revision humana obligatoria y el baseline sin modelo como camino operativo seguro). Estado del pipeline: REVIEW, porque fallan tres puertas: worst_group_recall (0.6531 < 0.70), recall_gap_pilot (0.1376 > 0.10) e intersectional_support (8 < 15). Tolerancia de equidad: brecha de recall entre grupos <= 0.10 y recall del grupo con peor desempeño >= 0.70. Escalamiento a STOP: si la brecha supera 0.15 o el recal

## 5. Interpretación y decisión final

**Observación.** La logística y el árbol alcanzan **la misma utilidad global** (recall 0,7174), pero
la logística tiene **brechas menores** (recall 0,1376 frente a 0,1813; FPR 0,1250 frente a 0,1683) y
ocupa **menos de la mitad** (3,56 KB frente a 7,90 KB). El grupo G1, con menor prevalencia del target
(49,7 % frente a 63,9 % de G2), recibe peor recall (0,653) y la cola prioriza más a G2 (tasa de
selección 0,678 frente a 0,446).

**Evidencia.** `reports/lab12/logistic/fairness_by_group.csv`,
`reports/lab12/logistic/global_metrics.json`, `reports/lab12/logistic/fairness_intersectional.csv` y
`reports/lab12/comparativa_alternativas.csv`, generadas por los scripts del profesor con corte
temporal y semilla fija.

**Interpretación.** La brecha **no prueba discriminación causal**: el modelo no usa
`protected_group` y una parte de la diferencia refleja la distinta prevalencia del target. Pero
tampoco puede descartarse: zona y banda de edad pueden portar el mismo sesgo de forma indirecta. En
las celdas interseccionales con menos de 15 observaciones (n=8, n=11 y n=14) el soporte es insuficiente
y esas cifras no son concluyentes. Con una sola partición y una sola semilla, sin intervalos de
confianza, las brechas son estimaciones puntuales.

**Decisión documentada: LIMITAR.** No hay despliegue autónomo: no se despliega de forma autónoma ni se
usa el ranking como orden de ejecución. Se autoriza un piloto en modo REVIEW con revisión humana
obligatoria, con la logística como única configuración, el baseline sin modelo como camino operativo
seguro y una puerta de equidad trimestral.

**Estado del pipeline: REVIEW.** Fallan tres puertas: `worst_group_recall` (0,6531 < 0,70),
`recall_gap_pilot` (0,1376 > 0,10) e `intersectional_support` (8 < 15).

**Escalamiento a STOP:** si la brecha de recall supera 0,15 o el recall del peor grupo baja de 0,60,
el sistema se retira al baseline sin modelo. Antes de cualquier uso con
datos reales hay que repetir la auditoría con un periodo posterior, con intervalos de confianza y con
el coste del daño cuantificado. El marco voluntario (NIST AI RMF) organiza el control; la
recomendación ética (alternativa no predictiva, apelación, condición de retiro) la decisión; y la
ausencia de una obligación legal aplicable al caso sintético no se sustituye con ninguna de las dos.